# Crime Prediction Model Prototyping

This notebook prototypes machine learning models for crime prediction:
1. Random Forest Classifier for crime type prediction
2. Linear Regression for crime count prediction
3. LSTM for time-series crime forecasting

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

# For LSTM
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    HAS_TENSORFLOW = True
except ImportError:
    HAS_TENSORFLOW = False
    print("TensorFlow not available. LSTM model will be skipped.")

import matplotlib.pyplot as plt
import pickle

print(f"TensorFlow available: {HAS_TENSORFLOW}")

In [ ]:
# Cell 2: Load data from database
DATABASE_URL = os.environ.get('DATABASE_URL', 'sqlite:///crime_data.db')
engine = create_engine(DATABASE_URL)

# Load crimes table
df = pd.read_sql_table('crimes', con=engine)
print(f"Loaded {len(df)} crime records")

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

# Display sample
df.head()

In [ ]:
# Cell 3: Data Preprocessing
# Create features for model training
df_model = df.copy()

# Extract time-based features
df_model['hour'] = pd.to_datetime(df_model['time'], format='%H:%M:%S', errors='coerce').dt.hour.fillna(0).astype(int)
df_model['day_of_week'] = df_model['date'].dt.dayofweek
df_model['month'] = df_model['date'].dt.month
df_model['year'] = df_model['date'].dt.year
df_model['is_weekend'] = (df_model['day_of_week'] >= 5).astype(int)

# Time of day categories
def get_time_category(hour):
    if 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    elif 18 <= hour < 24:
        return 'evening'
    else:
        return 'night'

df_model['time_category'] = df_model['hour'].apply(get_time_category)

# Encode categorical variables
le_type = LabelEncoder()
df_model['type_encoded'] = le_type.fit_transform(df_model['type'].fillna('Unknown'))

le_district = LabelEncoder()
df_model['district_encoded'] = le_district.fit_transform(df_model['district'].fillna('Unknown'))

le_time_cat = LabelEncoder()
df_model['time_cat_encoded'] = le_time_cat.fit_transform(df_model['time_category'])

print("Encoding mappings:")
print(f"Crime types: {dict(zip(le_type.classes_, range(len(le_type.classes_))))}")

# Prepare features and target
feature_cols = ['hour', 'day_of_week', 'month', 'is_weekend', 'district_encoded', 'time_cat_encoded']
if 'latitude' in df_model.columns:
    df_model['latitude'] = df_model['latitude'].fillna(df_model['latitude'].median())
    df_model['longitude'] = df_model['longitude'].fillna(df_model['longitude'].median())
    feature_cols.extend(['latitude', 'longitude'])

X = df_model[feature_cols].fillna(0)
y_type = df_model['type_encoded']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_type, test_size=0.2, random_state=42)

print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Cell 4: Train Random Forest Classifier
print("Training Random Forest Classifier for crime type prediction...")

rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_train)

print("Random Forest Classifier trained successfully!")

In [ ]:
# Cell 5: Evaluate Random Forest Classifier
# Predictions
y_pred = rf_classifier.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le_type.classes_))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_classifier.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Train Linear Regression for Crime Count Prediction
# Aggregate crimes by date for time-series regression
daily_crimes = df_model.groupby('date').agg({
    'id': 'count',
    'hour': 'mean',
    'day_of_week': 'first',
    'month': 'first',
    'is_weekend': 'first'
}).rename(columns={'id': 'crime_count'})

# Add lag features
daily_crimes['crime_count_lag1'] = daily_crimes['crime_count'].shift(1)
daily_crimes['crime_count_lag7'] = daily_crimes['crime_count'].shift(7)
daily_crimes['rolling_mean_7'] = daily_crimes['crime_count'].rolling(window=7).mean()

# Drop NaN from lag features
daily_crimes = daily_crimes.dropna()

# Prepare features and target for regression
reg_features = ['day_of_week', 'month', 'is_weekend', 'crime_count_lag1', 'crime_count_lag7', 'rolling_mean_7']
X_reg = daily_crimes[reg_features]
y_reg = daily_crimes['crime_count']

# Train/test split
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Regression Training samples: {len(X_reg_train)}")
print(f"Regression Test samples: {len(X_reg_test)}")

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_reg_train, y_reg_train)

print("\nLinear Regression model trained successfully!")

In [ ]:
# Cell 7: Evaluate Linear Regression
# Predictions
y_reg_pred = lr_model.predict(X_reg_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)

print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R²): {r2:.4f}")

# Plot predictions vs actual
plt.figure(figsize=(12, 6))
plt.plot(range(len(y_reg_test)), y_reg_test.values, label='Actual', alpha=0.7)
plt.plot(range(len(y_reg_test)), y_reg_pred, label='Predicted', alpha=0.7)
plt.xlabel('Sample Index')
plt.ylabel('Daily Crime Count')
plt.title('Linear Regression: Predicted vs Actual Daily Crime Counts')
plt.legend()
plt.tight_layout()
plt.show()

# Feature coefficients
print("\nFeature Coefficients:")
for feat, coef in zip(reg_features, lr_model.coef_):
    print(f"  {feat}: {coef:.4f}")

In [ ]:
# Cell 8: Prepare LSTM Data (30-day sequences)
if not HAS_TENSORFLOW:
    print("Skipping LSTM - TensorFlow not available")
else:
    # Use daily crime counts time series
    crime_series = daily_crimes['crime_count'].values
    
    # Create sequences
    SEQUENCE_LENGTH = 30
    X_lstm = []
    y_lstm = []
    
    for i in range(len(crime_series) - SEQUENCE_LENGTH):
        X_lstm.append(crime_series[i:i + SEQUENCE_LENGTH])
        y_lstm.append(crime_series[i + SEQUENCE_LENGTH])
    
    X_lstm = np.array(X_lstm)
    y_lstm = np.array(y_lstm)
    
    # Reshape for LSTM [samples, time steps, features]
    X_lstm = X_lstm.reshape((X_lstm.shape[0], X_lstm.shape[1], 1))
    
    # Train/test split
    split_idx = int(len(X_lstm) * 0.8)
    X_lstm_train = X_lstm[:split_idx]
    X_lstm_test = X_lstm[split_idx:]
    y_lstm_train = y_lstm[:split_idx]
    y_lstm_test = y_lstm[split_idx:]
    
    # Scale data
    scaler = StandardScaler()
    X_lstm_train_scaled = scaler.fit_transform(X_lstm_train.reshape(-1, 1)).reshape(X_lstm_train.shape)
    X_lstm_test_scaled = scaler.transform(X_lstm_test.reshape(-1, 1)).reshape(X_lstm_test.shape)
    
    print(f"LSTM Training sequences: {len(X_lstm_train)}")
    print(f"LSTM Test sequences: {len(X_lstm_test)}")

In [ ]:
# Cell 9: Build and Train LSTM Model
if not HAS_TENSORFLOW:
    print("Skipping LSTM - TensorFlow not available")
else:
    model_lstm = Sequential([
        LSTM(64, activation='relu', input_shape=(SEQUENCE_LENGTH, 1), return_sequences=True),
        Dropout(0.2),
        LSTM(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    
    model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
    
    print("LSTM Model Architecture:")
    model_lstm.summary()
    
    # Early stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    # Train
    print("\nTraining LSTM model...")
    history = model_lstm.fit(
        X_lstm_train_scaled, y_lstm_train,
        epochs=50,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=1
    )

In [ ]:
# Cell 10: Evaluate LSTM Model
if not HAS_TENSORFLOW:
    print("Skipping LSTM - TensorFlow not available")
else:
    # Predictions
    y_lstm_pred = model_lstm.predict(X_lstm_test_scaled).flatten()
    
    # Metrics
    lstm_rmse = np.sqrt(mean_squared_error(y_lstm_test, y_lstm_pred))
    lstm_r2 = r2_score(y_lstm_test, y_lstm_pred)
    
    print(f"LSTM Root Mean Squared Error (RMSE): {lstm_rmse:.4f}")
    print(f"LSTM R-squared (R²): {lstm_r2:.4f}")
    
    # Plot predictions vs actual
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    plt.plot(y_lstm_test, label='Actual', alpha=0.7)
    plt.plot(y_lstm_pred, label='Predicted', alpha=0.7)
    plt.xlabel('Time Step')
    plt.ylabel('Crime Count')
    plt.title('LSTM: Predictions vs Actual')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.title('LSTM Training History')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 11: Save Models
import os
from datetime import datetime

# Create models directory
models_dir = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'models')
os.makedirs(models_dir, exist_ok=True)

# Save Random Forest classifier
rf_path = os.path.join(models_dir, 'rf_classifier.pkl')
with open(rf_path, 'wb') as f:
    pickle.dump({'model': rf_classifier, 'label_encoder': le_type, 'feature_cols': feature_cols}, f)
print(f"Random Forest model saved to: {rf_path}")

# Save Linear Regression model
lr_path = os.path.join(models_dir, 'linear_regression.pkl')
with open(lr_path, 'wb') as f:
    pickle.dump({'model': lr_model, 'feature_cols': reg_features}, f)
print(f"Linear Regression model saved to: {lr_path}")

# Save LSTM model if available
if HAS_TENSORFLOW:
    lstm_path = os.path.join(models_dir, 'lstm_model.keras')
    model_lstm.save(lstm_path)
    print(f"LSTM model saved to: {lstm_path}")

# Summary
print("\n" + "="*50)
print("MODEL TRAINING SUMMARY")
print("="*50)
print(f"Random Forest Classifier - Accuracy: {accuracy:.4f}")
print(f"Linear Regression - RMSE: {rmse:.4f}, R²: {r2:.4f}")
if HAS_TENSORFLOW:
    print(f"LSTM - RMSE: {lstm_rmse:.4f}, R²: {lstm_r2:.4f}")